# SAIL-VL-2B -> GGUF: F16 + Q8_0 + Q4_K_M + mmproj + eval + auto HF upload
Self-sufficient notebook. Zero interactive prompts; any failure stops loudly.
Manual inputs: 1) Secrets: add HF_TOKEN. 2) Verify MODEL_ID and REPO_NAME below.
Runtime: Internet ON, CPU. ~25 min.

In [ ]:
import os, sys, time, json, hashlib, re, shutil, subprocess
W = "/kaggle/working"
os.chdir(W)
os.environ["PIP_NO_INPUT"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
# ---------------- SET THESE ----------------
MODEL_ID     = "BytedanceDouyinContent/SAIL-VL-2B"
REPO_NAME    = "ShayonSarker/SAIL-VL-2B-GGUF"
REPO_PRIVATE = False
MODEL_SLUG   = "sail-vl-2b"
QUANTS       = ["Q4_K_M", "Q8_0"]
# -------------------------------------------
N_THREADS = max(1, os.cpu_count() or 2)
print("threads:", N_THREADS, flush=True)

def run(cmd, log=None):
    print("$ " + cmd, flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=W, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        lines.append(line)
        print(line, end="", flush=True)
    p.wait()
    print("[exit %d in %.0fs]" % (p.returncode, time.time()-t0), flush=True)
    out = "".join(lines)
    if log:
        open(log, "w").write(out)
    if p.returncode != 0:
        raise RuntimeError("FAILED: " + cmd)
    return out

run("python3 --version && (nvidia-smi -L || echo no-gpu) && df -h /kaggle/working | tail -1")
run("pip install -q --no-cache-dir 'huggingface_hub>=1.3,<2.0' 'pillow>=10,<12'")

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN") or None
except Exception as e:
    print("secrets unavailable:", e)
    HF_TOKEN = None
print("HF_TOKEN:", "present" if HF_TOKEN else "MISSING")


In [ ]:
# [1] Clone llama.cpp, build
if os.path.exists("llama.cpp"):
    shutil.rmtree("llama.cpp")
run("git clone --depth 1 https://github.com/ggerganov/llama.cpp.git")
run("cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release && cmake --build build --config Release -j%d --target llama-quantize llama-mtmd-cli" % N_THREADS)
LLAMA = W + "/llama.cpp"
print("llama.cpp ready")


In [ ]:
# [2] Download model
MODEL_DIR = W + "/model"
if os.path.exists(MODEL_DIR):
    shutil.rmtree(MODEL_DIR)
run("python3 -c \"from huggingface_hub import snapshot_download; snapshot_download('%s', local_dir='%s', ignore_patterns=['*.md'])\"" % (MODEL_ID, MODEL_DIR))
assert os.path.exists(MODEL_DIR + "/config.json"), "Missing config.json"
assert os.path.exists(MODEL_DIR + "/tokenizer.json"), "Missing Qwen2 BPE tokenizer"
stale_sentencepiece = MODEL_DIR + "/tokenizer.model"
if os.path.exists(stale_sentencepiece):
    os.remove(stale_sentencepiece)
run("ls -la " + MODEL_DIR)


In [ ]:
# [3] Check architecture
config = json.load(open(MODEL_DIR + "/config.json"))
tokenizer_config = json.load(open(MODEL_DIR + "/tokenizer_config.json"))
arch = config.get("architectures", ["unknown"])[0]
vision_type = config.get("vision_config", {}).get("model_type", "?")
tokenizer_class = tokenizer_config.get("tokenizer_class")
assert tokenizer_class == "Qwen2Tokenizer", tokenizer_class
assert not os.path.exists(MODEL_DIR + "/tokenizer.model"), "Wrong tokenizer path selected"
print("Architecture:", arch), "| Vision:", vision_type, "| Tokenizer:", tokenizer_class


In [ ]:
# [4] Convert text model
OUT = W + "/output"
if os.path.exists(OUT):
    shutil.rmtree(OUT)
os.makedirs(OUT)

run("python3 %s/convert_hf_to_gguf.py %s --outfile %s/%s-f16.gguf --outtype f16" % (LLAMA, MODEL_DIR, OUT, MODEL_SLUG))
run("%s/build/bin/llama-quantize %s/%s-f16.gguf %s/%s-Q8_0.gguf Q8_0" % (LLAMA, OUT, MODEL_SLUG, OUT, MODEL_SLUG))
run("%s/build/bin/llama-quantize %s/%s-f16.gguf %s/%s-Q4_K_M.gguf Q4_K_M" % (LLAMA, OUT, MODEL_SLUG, OUT, MODEL_SLUG))
print("Text model converted")


In [ ]:
# [5] Convert mmproj
preprocessor_path = MODEL_DIR + "/preprocessor_config.json"
if not os.path.exists(preprocessor_path):
    json.dump({"image_mean": [0.485, 0.456, 0.406], "image_std": [0.229, 0.224, 0.225]}, open(preprocessor_path, "w"))
run("python3 %s/convert_hf_to_gguf.py %s --outfile %s/mmproj-%s-f16.gguf --outtype f16 --mmproj" % (LLAMA, MODEL_DIR, OUT, MODEL_SLUG))
print("mmproj converted")


In [ ]:
# [6] Verify GGUF files
for f in sorted(os.listdir(OUT)):
    if f.endswith(".gguf"):
        path = OUT + "/" + f
        with open(path, "rb") as fp:
            magic = fp.read(4)
        size_mb = os.path.getsize(path) / 1e6
        ok = magic == b'GGUF'
        print("%s  %.1f MB  %s" % (f, size_mb, "OK" if ok else "BAD"))
        assert ok, "Invalid GGUF: " + f


In [ ]:
# [7] Inference test (gate)
test_image = MODEL_DIR + "/statics/14.jpg"
assert os.path.exists(test_image), "Missing bundled test image"
cmd = "%s/build/bin/llama-mtmd-cli -m %s/%s-Q8_0.gguf --mmproj %s/mmproj-%s-f16.gguf --image %s -p 'Describe the image and read any visible text.' --temp 0 -t %d -n 128" % (LLAMA, OUT, MODEL_SLUG, OUT, MODEL_SLUG, test_image, N_THREADS)
print("$ " + cmd, flush=True)
p = subprocess.run(cmd, shell=True, cwd=W, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print(p.stderr, end="")
print(p.stdout, end="")
if p.returncode != 0:
    raise RuntimeError("FAILED: " + cmd)
test_out = p.stdout.strip()
normalized = re.sub(r"[^a-z0-9]+", "", test_out.lower())
expected_terms = ("french", "bulldog", "dog", "canine", "monday", "deck", "wood", "platform")
has_text = len(normalized) >= 20 and any(term in test_out.lower() for term in expected_terms)
print("Gate output:", repr(test_out))
print("Gate:", "PASS" if has_text else "FAIL")
if not has_text:
    raise SystemExit("INFERENCE GATE FAILED: no semantic image description")


In [ ]:
# [8] Pack artifacts + SHA256
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

ARTIFACTS = ["%s-Q8_0.gguf" % MODEL_SLUG, "%s-Q4_K_M.gguf" % MODEL_SLUG, "%s-f16.gguf" % MODEL_SLUG, "mmproj-%s-f16.gguf" % MODEL_SLUG]
for f in ARTIFACTS:
    assert os.path.exists(OUT + "/" + f) and os.path.getsize(OUT + "/" + f) > 0, "missing " + f
shutil.copy2(MODEL_DIR + "/LICENSE", OUT + "/LICENSE")

SHAS = {f: sha256_file(OUT + "/" + f) for f in ARTIFACTS}
open(OUT + "/SHA256.txt", "w").write("".join(SHAS[f] + "  " + f + "\n" for f in ARTIFACTS))
llama_commit = subprocess.check_output(["git", "-C", LLAMA, "rev-parse", "HEAD"], text=True).strip()
repro = {"model_id": MODEL_ID, "repo": REPO_NAME, "quants": QUANTS,
          "llama_cpp_commit": llama_commit, "inference_output": test_out,
          "sha": {f: SHAS[f][:16] for f in ARTIFACTS},
          "sizes_mb": {f: round(os.path.getsize(OUT+"/"+f)/1e6, 1) for f in ARTIFACTS}}
json.dump(repro, open(OUT + "/REPRO.json", "w"), indent=2)
run("ls -lh " + OUT)


In [ ]:
# [9] Auto-upload to HF
if not HF_TOKEN:
    raise SystemExit("HF_TOKEN missing")

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
owner = api.whoami()["name"]
assert REPO_NAME.startswith(owner + "/"), "REPO_NAME must be owned by the HF_TOKEN user"
repo_id = REPO_NAME
api.create_repo(repo_id, repo_type="model", exist_ok=True, private=REPO_PRIVATE)

UPLOAD_FILES = ARTIFACTS + ["SHA256.txt", "REPRO.json", "LICENSE"]
for f in UPLOAD_FILES:
    print("Uploading", f, "...")
    api.upload_file(path_or_fileobj=OUT+"/"+f, path_in_repo=f, repo_id=repo_id,
                    repo_type="model", commit_message="add " + f)
print("UPLOADED -> https://huggingface.co/" + repo_id)
